# Memory management and large trajectories

MolSysMT can work with trajectories that are too large to fit in RAM. This page explains how the
library decides when to switch to chunked processing, how to control that decision, and what to
expect from each mode.

## The memory wall problem

Most structural analysis functions in MolSysMT work by loading the full coordinate array into RAM
and running a kernel over it. This **eager path** is fast and simple, but it has a hard limit:
if the trajectory is larger than available memory the process crashes.

A single-precision trajectory with 1 million atoms and 10 000 frames already occupies roughly
120 GB. Even with 64 GB of RAM, many realistic production trajectories exceed what the eager
path can handle.

MolSysMT addresses this with a **heavy path**: a chunked execution mode that reads the trajectory
in pieces, passes each piece to an analysis kernel, and accumulates partial results until the full
answer is ready. The public API does not change — the same function call is used whether the
trajectory fits in memory or not.

## The `heavy_mode` parameter

All analysis functions that support chunked execution accept a `heavy_mode` argument with three
possible values:

| Value | Behaviour |
|---|---|
| `'auto'` (default) | Estimate the memory footprint. Use the eager path if it fits within `molsysmt.configure.max_ram_usage`; otherwise switch to the heavy path. |
| `'force'` | Always use the heavy (chunked) path, even for small data. Useful for testing or for guaranteeing bounded memory consumption regardless of trajectory size. |
| `'off'` | Always use the eager path. Useful when you know the data fits in RAM and want to skip the overhead of the decision logic. |

Example — using `get_center` with explicit mode control:

```python
import molsysmt as msm

# auto: MolSysMT decides
c = msm.structure.get_center('trajectory.h5msm', selection='all')

# force heavy path regardless of trajectory size
c = msm.structure.get_center('trajectory.h5msm', selection='all', heavy_mode='force')

# force eager path
c = msm.structure.get_center('trajectory.h5msm', selection='all', heavy_mode='off')
```

## Configuration

The automatic decision (when `heavy_mode='auto'`) depends on configuration values in
`molsysmt.configure`:

```python
import molsysmt as msm

# RAM threshold (bytes).
# Default: 50% of the machine's total physical RAM (detected at import time).
# If the estimated footprint of eager loading exceeds this, the heavy path is used.
msm.configure.max_ram_usage

# Frames per chunk. Default: 100.
# Controls how many frames are loaded at once in the heavy path.
msm.configure.chunk_size

# Global default mode applied when heavy_mode is not passed to a function.
# Default: 'auto'. Can be set to 'force' or 'off' to change the default for all calls.
msm.configure.heavy_mode

# Whether to emit progress and decision events through SMonitor. Default: True.
msm.configure.emit_heavy_telemetry

# Memory pressure warning threshold (fraction of max_ram_usage). Default: 0.80.
# A MemoryPressureWarning is emitted when the process RSS exceeds this fraction.
msm.configure.memory_pressure_threshold
```

You can change them for the current session:

```python
msm.configure.max_ram_usage = 8 * 1024**3   # 8 GB threshold
msm.configure.chunk_size = 200
msm.configure.emit_heavy_telemetry = False   # quiet batch runs
msm.configure.memory_pressure_threshold = 0.90  # raise the warning threshold
```

The footprint estimate formula is:

```
estimated_bytes = n_atoms × n_structures × 3 × 8 × 1.20
```

(float64 coordinates with a 20 % safety margin for Python overhead and temporary arrays.)

## Supported operations and forms

Not every analysis function and not every molecular-system form supports the heavy path yet.
The following combinations are supported in MolSysMT 1.0:

**Functions:**
- `molsysmt.structure.get_center`
- `molsysmt.structure.get_rmsd`
- `molsysmt.structure.get_distances`

**Forms:**

| Form | Description |
|---|---|
| `file:h5msm` | File path to an `.h5msm` trajectory file |
| `molsysmt.H5MSMFileHandler` | Open file handler (avoids reopening for each call) |
| `file:xtc` | GROMACS XTC trajectory file |

If you request `heavy_mode='force'` on a form or operation combination that does not support
it, MolSysMT raises an explicit `UnsupportedHeavyOperationError` with a diagnostic message.
It will never silently fall back to a broken result.

## Large outputs: PersistentResultHandle

Solving the *input* memory wall is not enough if the *output* of the analysis is itself too large
to fit in RAM. For example, pairwise distances for 10 000 atoms over 50 000 frames occupy
roughly 400 GB.

When the predicted output size exceeds `molsysmt.configure.max_ram_usage`, MolSysMT automatically
returns a `PersistentResultHandle` instead of a plain NumPy array. The handle is backed by a
temporary disk file (NumPy memmap) and behaves like an array for indexing and slicing:

```python
from molsysmt.structure import get_distances

result = get_distances('big_trajectory.h5msm', selection='all')

# result may be a PersistentResultHandle if the output is very large
print(type(result))          # PersistentResultHandle or puw quantity
print(result.shape)          # (n_structures, n_atoms, n_atoms)

# Access frames normally
frame_0 = result[0]          # loads only one frame from disk

# Copy the full result into RAM when you are ready
full = result.to_memory()

# Flush pending writes to disk
result.flush()

# Clean up the temporary file when done
result.cleanup()
```

A `PersistentResultHandle` can also be used as a context manager to ensure cleanup:

```python
with get_distances('big_trajectory.h5msm', selection='all') as result:
    frame_0 = result[0]
# backing file deleted automatically on exit
```

MolSysMT also checks available disk space before creating the handle. If there is not enough room,
it raises `HeavyOutputFailureError` before any computation starts (a 10% free-space margin is
reserved by default).
### Controlling the output file location

By default, MolSysMT creates the backing memmap in a system-managed temporary file that is
deleted automatically when `cleanup()` is called. If you want to keep the file — for example to
pass it to another process or to persist results across sessions — you can specify a path:

```python
result = get_distances(
    'big_trajectory.h5msm',
    selection='all',
    output_path='/scratch/distances_output.dat',
)

# The file at /scratch/distances_output.dat is NOT deleted by cleanup()
result.cleanup()
```

Parent directories are created automatically if they do not exist. The lifecycle of the file is
entirely your responsibility when you provide a path.

## Telemetry and progress

When `molsysmt.configure.emit_heavy_telemetry` is `True` (the default), the heavy path emits
structured events through the SMonitor system:

| Event code | Name | Payload highlights |
|---|---|---|
| `MSM-INFO-HVY-001` | `HeavyPathSelected` | operation, form, footprint\_bytes, max\_ram\_usage |
| `MSM-INFO-HVY-002` | `EagerPathAccepted` | operation, footprint\_bytes |
| `MSM-INFO-HVY-003` | `ChunkProcessed` | chunk\_index, n\_chunks, elapsed\_s, eta\_s |
| `MSM-WARN-HVY-001` | `SlowChunkIOWarning` | chunk\_index, io\_time\_s |
| `MSM-WARN-HVY-002` | `CorruptFrameSkippedWarning` | chunk\_index, frame\_index, reason |
| `MSM-WARN-HVY-003` | `MemoryPressureWarning` | chunk\_index, rss\_bytes, budget\_bytes, pressure\_pct |
| `MSM-ERROR-HVY-001` | `UnsupportedHeavyOperationError` | operation, form, reason |
| `MSM-ERROR-HVY-002` | `HeavyOutputFailureError` | reason, predicted\_bytes, available\_bytes |

You can disable telemetry for quiet batch runs:

```python
msm.configure.emit_heavy_telemetry = False
```

See [Logging and diagnostics](molsysmt_logging_user_guide.md) for how to capture or filter
these events.

## Practical tips

**Reuse an open handler.** Opening the `.h5msm` file once and passing the
`molsysmt.H5MSMFileHandler` object to multiple function calls is more efficient than
passing the file path each time, because the heavy path can reuse the already-open HDF5 file:

```python
h = msm.convert('trajectory.h5msm', to_form='molsysmt.H5MSMFileHandler')
try:
    center = msm.structure.get_center(h, selection='all', heavy_mode='force')
    rmsd   = msm.structure.get_rmsd(h, selection='atom_type!="H"', heavy_mode='force')
finally:
    h.close()
```

**Tune `chunk_size` for your storage.** Larger chunks reduce overhead but require more RAM.
A chunk size between 200 and 1000 frames works well for most workloads on local SSDs.

**`heavy_mode='force'` for reproducible benchmarks.** Using `heavy_mode='force'` on a small
test trajectory guarantees you are exercising the same code path as a production run with a large
one. The parity tests in the test suite use this pattern.

**PBC and distances.** `get_distances` with `pbc=True` requests the box vectors alongside
coordinates in each chunk. This requires that the form declares `'box': True` in its
`_heavy_support` dict. All currently supported heavy-mode forms (`file:h5msm`,
`molsysmt.H5MSMFileHandler`, `file:xtc`) declare this support.